# day-23-function-calling — worked solutions & answer key

Solutions to the exercises in [`../lesson.ipynb`](../lesson.ipynb), plus the self-check answer key. **Try each exercise yourself first** — the value is in the attempt, not the answer.

In [8]:
# ---- Solution 2 ----
def get_weather_v2(city, units="celsius"):
    if city == "Atlantis": return {"error": "city not found"}
    return get_weather(city, units)
TOOLS["get_weather"]["fn"] = get_weather_v2

class RecoveringModel(MockToolModel):
    def create(self, **kw):
        msgs = kw["messages"]
        errs = [b for m in msgs if m["role"] == "user" and isinstance(m["content"], list)
                for b in m["content"] if isinstance(b, dict) and b.get("is_error")]
        if errs:
            return Message(content=[TextBlock("That city isn't available. [would ask user for another]")],
                           stop_reason="end_turn")
        return super().create(**kw)
client.messages = RecoveringModel()

def run_with_errors(q):
    messages = [{"role": "user", "content": q}]
    for _ in range(4):
        r = client.messages.create(model="m", max_tokens=500,
                                   tools=[t["schema"] for t in TOOLS.values()], messages=messages)
        messages.append({"role": "assistant", "content": r.content})
        if r.stop_reason != "tool_use":
            return "".join(b.text for b in r.content if b.type == "text")
        res = []
        for b in r.content:
            if b.type == "tool_use":
                out = TOOLS[b.name]["fn"](**b.input)
                res.append({"type": "tool_result", "tool_use_id": b.id,
                            "content": json.dumps(out), "is_error": "error" in out})
        messages.append({"role": "user", "content": res})
    return "stopped"

print("S2:", run_with_errors("what's the weather in Atlantis"))
client.messages = MockToolModel()

S2: That city isn't available. [would ask user for another]


In [9]:
# ---- Solution 6 ----
def schema_token_cost(n_tools, tok_each, turns):
    return n_tools * tok_each * turns
raw = schema_token_cost(10, 80, 20)
print(f"S6: 10 tools x 80 tok x 20 turns = {raw:,} input tokens just for schemas")
print(f"    with prompt caching (tools are in the cached prefix): ~{raw*0.1:,.0f} effective")
print("    with tool search (defer_loading + a search tool): only the ~3 relevant schemas load")

S6: 10 tools x 80 tok x 20 turns = 16,000 input tokens just for schemas
    with prompt caching (tools are in the cached prefix): ~1,600 effective
    with tool search (defer_loading + a search tool): only the ~3 relevant schemas load


### Solutions 1, 3, 4, 5 (sketch)

**S1:** a description of just "weather" gives the model nothing to distinguish it from, say, a
`get_forecast` tool — it calls the wrong one or asks for clarification. Good descriptions state
purpose, trigger phrases, and boundaries. This is the highest-leverage thing in a tool spec.

**S3:** `PERSON = {"type":"object","additionalProperties":False,"properties":{"name":{"type":
"string"},"age":{"type":"integer"},"emails":{"type":"array"}},"required":["name","age"]}`;
extend `validate_json` with an `"array"` case. `{"name":"A","age":"twelve"}` → "age: expected
integer"; `{"name":"A","age":1,"nickname":"x"}` → "unexpected keys".

**S4:** e.g. "convert my Tokyo weather-in-degrees to fahrenheit" — the model must first call
`get_weather`, *see the number*, then... actually that's contrived; a real dependent case is
"look up team X's budget, then convert it to EUR". The model emits `get_team` in turn 1, reads
the result, emits `convert_currency` in turn 2. Two API calls minimum; parallel only works for
*independent* calls.

**S5:** forcing `convert_currency` is useful when you *know* the next step is a conversion and
want to skip the model deciding — e.g. a UI button. It breaks on Fable 5.1 / Mythos 5.1 (400);
there you use `auto` + "call convert_currency now" in the prompt, or structured outputs if you
only wanted the JSON.

### Answer key
1. No. It emits a `tool_use` block proposing a call with arguments; you execute the function
   and return the result as a `tool_result` block.
2. (1) `user`: question + you pass `tools`. (2) `assistant`: `tool_use` block(s),
   `stop_reason="tool_use"` — you append this whole turn to `messages`. (3) `user`:
   `tool_result` block(s) with matching `tool_use_id`. (4) `assistant`: final text,
   `stop_reason="end_turn"`.
3. Splitting them across multiple `user` messages trains the model to stop making parallel
   calls — it learns that only one result comes back per turn.
4. That `tool_use.input` validates the schema exactly (all required fields present, no extra
   keys, correct types). It's a top-level field on the **tool definition**, not on
   `tool_choice`, and needs `additionalProperties: false` + `required`.
5. When you want the model's *answer* as schema-valid JSON (classification, extraction), not a
   call to your code — and especially on models where forced `tool_choice` is rejected.
6. `{"type": "any"}` and `{"type": "tool", "name": ...}` → 400. Use `{"type": "auto"}` plus a
   prompt instruction naming the tool, `strict: true` for valid args, or structured outputs.
7. Prompt-cache the prefix so the schemas are cached (~10% cost on hits); use tool search
   (`defer_loading: true` + a search tool) so only the relevant schemas load per request.
   Also: fewer, better tools.